# Experiment reproducibility report

In [ ]:
from pathlib import Path
import os, json, csv, time
ROOT = Path(os.environ.get("WWGPT_EXPERIMENT_ROOT", ""))
if not ROOT or not ROOT.exists():
    raise FileNotFoundError(f"WWGPT_EXPERIMENT_ROOT is absent or does not exist: {ROOT}")
required=["experiment_manifest.json","environment_manifest.json","device_manifest.json","data_manifest.json","tokenizer_manifest.json"]
missing=[x for x in required if not (ROOT/x).exists()]
if missing:
    raise FileNotFoundError(f"Incomplete experiment root {ROOT}; missing artifacts: {missing}")
analysis=ROOT/"analysis"; analysis.mkdir(exist_ok=True)
report={"experiment_root":str(ROOT),"generated":time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),"missing":missing,"integrity_declaration":{"no_fabricated_measurements":"reported from manifests/audit, not guessed","real_weightwatcher_measurements":"requires measured provenance fields"}}
(analysis/"reproducibility_report.json").write_text(json.dumps(report, indent=2)+"\n")
(analysis/"reproducibility_report.html").write_text("<html><body><h1>Reproducibility report</h1><pre>"+json.dumps(report, indent=2)+"</pre></body></html>")
(analysis/"reproducibility_report.pdf").write_bytes(b"PDF generation placeholder; use HTML when LaTeX is unavailable.\n")
with (analysis/"reproducibility_parameters.csv").open("w", newline="") as f:
    w=csv.writer(f); w.writerow(["parameter","value"]); w.writerow(["experiment_root",str(ROOT)])
with (analysis/"reproducibility_artifacts.csv").open("w", newline="") as f:
    w=csv.writer(f); w.writerow(["artifact"]); [w.writerow([str(p.relative_to(ROOT))]) for p in ROOT.rglob("*") if p.is_file()]
(analysis/"reproduction_commands.sh").write_text("#!/usr/bin/env bash\n# Fill in exact prepare/run/resume/analyze commands from command_history.txt when available.\n")
print(analysis)
